In [2]:
import spacy
from nltk.corpus import stopwords
import re
import os
import unicodedata
from unidecode import unidecode
import pandas as pd
from spacy.tokens import Doc

In [3]:
import os
from dotenv import load_dotenv
from pathlib import Path

# Como o notebook está em 'corretor/src/3_avaliacao' e o .env em 'corretor',
# precisamos voltar duas pastas (../..) para achar o arquivo.
env_path = Path('../../.env')

# Carrega o arquivo explicitamente
load_dotenv(dotenv_path=env_path)

True

In [4]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\joaol\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
nlp = spacy.load("pt_core_news_lg")

In [6]:
PATH_BASE_TESTES_FINAIS = os.getenv('PATH_BASE_TESTES_FINAIS')
PATH_GABARITO_CORRETOR = os.getenv('PATH_GABARITO_CORRETOR')
PATH_DADOS_ANONIMIZACAO_SPACY = os.getenv('PATH_DADOS_ANONIMIZACAO_SPACY')

In [7]:
data = pd.read_csv(PATH_BASE_TESTES_FINAIS)

In [8]:
def preprocess(relato):
    def remove_stop_words(relato):
        sw = stopwords.words('portuguese')
        sw = [unidecode(word).upper() for word in sw]
        return ' '.join([token for token in re.split("[, ]", relato) if len(token) > 2 and token not in sw])
    
    def clean_relato(relato):
        relato = re.sub(r'\@(\w*\d?)', " ", relato)
        relato = re.sub(r'([A-z1-9]*_+[A-z1-9]*)|(_+[A-z1-9]*)|([A-z1-9]*_+)', " ", relato)
        relato = re.sub(r"((?:HTTPS?|WWW)(?:\:\/\/?)?[*a-zA-Z0-9\.\/\?\:@\-_=#]+[\.][a-zA-Z]{2,6}[a-zA-Z0-9\.\&\/\?\:@\-_=#]*)", " ", relato)
        relato = re.sub(r'(<[^<>]*>)|(&NBSP;)|(&nbsp;)', " ", relato)
        relato = re.sub(r"[^\w\s]", " ", relato)
        relato = re.sub(r"[\d]", " ", relato)
        relato = remove_stop_words(relato)
        return relato
    
    relato = clean_relato(relato)
    return relato

def normalize_text(text):
    if pd.isna(text):
        return text
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return text.upper()

In [9]:
data["relato_tratado"] = data["relato"].apply(normalize_text).apply(preprocess)

In [10]:
data['relato_tratado'][230]

'SUBJETIVO ADMISSAO DIABETES MELLITUS TIPO CENTRO DIABETES ENDOCRINOLOGIA DATA IDENTIFICACAO NOME CARMEN LUCIA GOMES IDADE DOMINANCIA MANUAL DESTRO CANHOTO AMBIDESTRO SEI ESCOLARIDADE ENSINO MEDIO COMPLETO TRABALHA SECRETARIA PROCEDENTE OSASCO PAULO EMAIL CARMEN GOMES HIPOTESES DIAGNOSTICAS DIABETES MELLITUS TIPO MAL CONTROLADA NECESSIDADE INTENSIFICACAO TRATAMENTO FARMACOLOGICO FARMACOLOGICO COMORBIDADES CLINICAS HIPERTENSAO ARTERIAL SISTEMICA HIPERCOLESTEROLEMIA COMORBIDADES NEUROLOGICAS PSIQUIATRICAS NEGA ANTECEDENTES HISTORIA FAMILIAR SIM SEI MAE IRMAO DOENCA RENAL CRONICA SIM SEI ESTAGIO RETINOPATIA DIABETICA SIM SEI GRAVIDADE NEUROPATIA PERIFERICA SIM SEI SENSITIVA MOTORA AUTONOMICA COMPLICACOES AGUDAS HISTORICO CETOACIDOSE DIABETICA ANOS MEDICACOES MEDICACOES ANTIDIABETICAS ATUAIS DOSES METFORMINA GLICLAZIDA INSULINA NPH TRATAMENTOS UTILIZADOS METFORMINA SULFONILUREIA GLICLAZIDA DPP INIBIDORES GLP ANALOGOS SGLT INIBIDORES INSULINA ACAO RAPIDA INSULINA BASAL NPH TIAZOLIDINEDIONAS

In [11]:
data['relato_tratado'] = data['relato_tratado'].str.lower()

In [12]:
base_relato_teste = data.copy()

In [13]:
def check_names(output_list : pd.DataFrame):
    gab = pd.read_csv(PATH_GABARITO_CORRETOR)
    copia_output_list = output_list.copy()
    
    copia_output_list = copia_output_list[copia_output_list['y_palavra'] != 2]
    copia_output_list['y_palavra'] = copia_output_list['y_palavra'].astype(int)

    merge_gab_output_list = copia_output_list.merge(gab[['palavra', 'y']], left_on='palavra', right_on='palavra', how='left')
    merge_gab_output_list['y'] = merge_gab_output_list['y'].fillna(0).astype(int)
    merge_gab_output_list = merge_gab_output_list[merge_gab_output_list['y_palavra'] != 2]

    return merge_gab_output_list

In [14]:
for idx, text in base_relato_teste["relato_tratado"].items():
    resultados = []
    doc = nlp(text)
    for token in doc:
        label = 1 if token.ent_type_ == "PER" else 0
        resultados.append((token.text, label))
    
    df_resultado = pd.DataFrame(resultados, columns=["palavra", "y_palavra"])
    df_resultado['palavra'] = df_resultado['palavra'].str.upper()
    pred_gab_names = check_names(df_resultado)
    base_relato_teste.at[idx, 'spacy_qtd_tokens'] = len(df_resultado['palavra'])
    base_relato_teste.at[idx, 'spacy_qtd_pred_nomes'] = len(pred_gab_names[pred_gab_names['y_palavra'] == 1])
    base_relato_teste.at[idx, 'spacy_qtd_gab_nomes'] = len(pred_gab_names[pred_gab_names['y'] == 1])
    base_relato_teste.at[idx, 'spacy_nomes_acertados'] = len(pred_gab_names[(pred_gab_names['y_palavra'] == 1) & (pred_gab_names['y'] == 1)])
    print(f"{idx} de 230")

0 de 230
1 de 230
2 de 230
3 de 230
4 de 230
5 de 230
6 de 230
7 de 230
8 de 230
9 de 230
10 de 230
11 de 230
12 de 230
13 de 230
14 de 230
15 de 230
16 de 230
17 de 230
18 de 230
19 de 230
20 de 230
21 de 230
22 de 230
23 de 230
24 de 230
25 de 230
26 de 230
27 de 230
28 de 230
29 de 230
30 de 230
31 de 230
32 de 230
33 de 230
34 de 230
35 de 230
36 de 230
37 de 230
38 de 230
39 de 230
40 de 230
41 de 230
42 de 230
43 de 230
44 de 230
45 de 230
46 de 230
47 de 230
48 de 230
49 de 230
50 de 230
51 de 230
52 de 230
53 de 230
54 de 230
55 de 230
56 de 230
57 de 230
58 de 230
59 de 230
60 de 230
61 de 230
62 de 230
63 de 230
64 de 230
65 de 230
66 de 230
67 de 230
68 de 230
69 de 230
70 de 230
71 de 230
72 de 230
73 de 230
74 de 230
75 de 230
76 de 230
77 de 230
78 de 230
79 de 230
80 de 230
81 de 230
82 de 230
83 de 230
84 de 230
85 de 230
86 de 230
87 de 230
88 de 230
89 de 230
90 de 230
91 de 230
92 de 230
93 de 230
94 de 230
95 de 230
96 de 230
97 de 230
98 de 230
99 de 230
100 de 230

In [15]:
base_relato_teste

,n_bop,relato,nomes,relato_tratado,spacy_qtd_tokens,spacy_qtd_pred_nomes,spacy_qtd_gab_nomes,spacy_nomes_acertados
0,1,"SUBJETIVO NEFROLOGIA 220 A - ALICE MORAES , 37...","ALICE, MORAES, PAULO, SOUSA, MATOS",subjetivo nefrologia alice moraes anos profess...,84.0,13.0,9.0,8.0
1,2,SUBJETIVO FISIOTERAPIA 402 A - RAFAELA MARTINS...,"RAFAELA, MARTINS, MARIA, SILVA, FERNANDO, CRUZ...",subjetivo fisioterapia rafaela martins anos de...,88.0,12.0,10.0,9.0
2,3,SUBJETIVO CARDIOLOGIA 205 A - RICARDO GONÇALVE...,"RICARDO, GONÇALVES, ANDRÉ, FERNANDA, LIMA, OLI...",subjetivo cardiologia ricardo goncalves anos a...,91.0,8.0,7.0,5.0
3,4,SUBJETIVO GASTROENTEROLOGIA PACIENTE : PEDRO S...,"PEDRO, SANTOS, LUISA, FERNANDES",subjetivo gastroenterologia paciente pedro san...,76.0,5.0,5.0,4.0
4,5,"SUBJETIVO ORTOPEDIA 78 A - JOANA MARTINS , 52 ...","JOANA, MARTINS, RENATA, COSTA",subjetivo ortopedia joana martins anos profess...,81.0,11.0,9.0,8.0
...,...,...,...,...,...,...,...,...
1881,1882,SUBJETIVO UER - SALA LARANJA ID : CLÁUDIA SOAR...,"CLÁUDIA, SOARES, ALMEIDA, PEDRO, LARISSA, GOME...",subjetivo uer sala laranja claudia soares alme...,151.0,22.0,20.0,16.0
1882,1883,SUBJETIVO UER - SALA AMARELA ID : RENATO AUGUS...,"RENATO, AUGUSTO, MOREIRA, FERNANDA, LOPES, SAM...",subjetivo uer sala amarela renato augusto more...,166.0,26.0,15.0,15.0
1883,1884,SUBJETIVO UER - SALA AMARELA ID : GERTRUDES SI...,"GERTRUDES, SILVA, CARNEIRO, CLARISSA, DIEGO, H...",subjetivo uer sala amarela gertrudes silva car...,158.0,22.0,17.0,9.0
1884,1885,SUBJETIVO CENTRO DE SAÚDE - CONSULTÓRIO ID : E...,"ELIANA, COSTA, SILVA, MARIANA, FREITAS, RAFAEL...",subjetivo centro saude consultorio eliana cost...,123.0,21.0,18.0,16.0


In [16]:
base_relato_teste.to_csv(PATH_DADOS_ANONIMIZACAO_SPACY, index=False)